<a href="https://colab.research.google.com/github/rafiwinno/SmartDiet-Assistant/blob/main/Feature_Nutrition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#FEATURE ENGINEERING

In [44]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

###Load Data

In [45]:
df = pd.read_csv('cleaned_nutrition.csv')
df.columns = df.columns.str.lower().str.strip().str.replace(' ', '_')

In [46]:
print("✅ Dataset loaded")
print(f"   Shape : {df.shape}")
print(f"   Kolom : {df.columns.tolist()}")

✅ Dataset loaded
   Shape : (3454, 15)
   Kolom : ['vitamin_c_(mg_per_100g)', 'vitamin_b11_(mg_per_100g)', 'sodium_(mg_per_100g)', 'calcium_(mg_per_100g)', 'carbohydrates_(g_per_100g)', 'food', 'iron_(mg_per_100g)', 'calories_(kcal_per_100g)', 'sugars_(g_per_100g)', 'dietary_fiber_(g_per_100g)', 'fat_(g_per_100g)', 'protein_(g_per_100g)', 'food_normalized', 'kalori_kategori', 'category']


###Kalkulasi Medis

In [47]:
# FUNGSI KALKULASI MEDIS

ACTIVITY_MULTIPLIER = {
    'sedentary'        : 1.2,
    'lightly_active'   : 1.375,
    'moderately_active': 1.55,
    'very_active'      : 1.725,
    'extra_active'     : 1.9
}

MACRO_RATIO = {
    'lose'    : {'protein': 0.35, 'fat': 0.30, 'carbs': 0.35},
    'maintain': {'protein': 0.30, 'fat': 0.30, 'carbs': 0.40},
    'gain'    : {'protein': 0.30, 'fat': 0.25, 'carbs': 0.45}
}

GOAL_ADJUSTMENT = {
    'lose'    : -500,
    'maintain': 0,
    'gain'    : +300
}

def calculate_bmr(weight_kg, height_cm, age, gender):
    if gender.lower() == 'male':
        return round(88.362 + (13.397 * weight_kg) + (4.799 * height_cm) - (5.677 * age), 2)
    else:
        return round(447.593 + (9.247 * weight_kg) + (3.098 * height_cm) - (4.330 * age), 2)

def calculate_tdee(bmr, activity_level):
    return round(bmr * ACTIVITY_MULTIPLIER[activity_level], 2)

def calculate_calorie_goal(tdee, goal):
    # Hard boundary minimum 1200 kcal (project plan risk management)
    return max(round(tdee + GOAL_ADJUSTMENT[goal], 2), 1200.0)

def calculate_macros(calorie_goal, goal):
    r = MACRO_RATIO[goal]
    return {
        'protein_target_g': round((calorie_goal * r['protein']) / 4, 1),
        'fat_target_g'    : round((calorie_goal * r['fat'])     / 9, 1),
        'carbs_target_g'  : round((calorie_goal * r['carbs'])   / 4, 1),
    }

###USER PROFILE

In [48]:
# GENERATE SYNTHETIC USER PROFILES (Untuk membuat training data untuk AI Engineer
# Karena tidak ada data user nyata, kita generate kombinasi yang realistis secara medis

np.random.seed(42)
N = 500  # jumlah sampel sintetis

genders     = ['male', 'female']
activities  = list(ACTIVITY_MULTIPLIER.keys())
goals       = list(GOAL_ADJUSTMENT.keys())

synthetic_users = pd.DataFrame({
    'gender'        : np.random.choice(genders, N),
    'weight_kg'     : np.where(
                        np.random.choice(['male','female'], N) == 'male',
                        np.random.uniform(55, 100, N),   # range BB pria
                        np.random.uniform(45, 85,  N)    # range BB wanita
                      ),
    'height_cm'     : np.where(
                        np.random.choice(['male','female'], N) == 'male',
                        np.random.uniform(160, 185, N),  # range TB pria
                        np.random.uniform(150, 170, N)   # range TB wanita
                      ),
    'age'           : np.random.randint(18, 60, N),
    'activity_level': np.random.choice(activities, N),
    'goal'          : np.random.choice(goals, N),
})

# Bulatkan BB dan TB
synthetic_users['weight_kg'] = synthetic_users['weight_kg'].round(1)
synthetic_users['height_cm'] = synthetic_users['height_cm'].round(1)

print(f"Synthetic user profiles: {synthetic_users.shape}")
print(synthetic_users.head(3))

Synthetic user profiles: (500, 6)
   gender  weight_kg  height_cm  age activity_level  goal
0    male       86.4      151.9   48      sedentary  gain
1  female       79.1      168.2   45      sedentary  gain
2    male       68.9      163.4   28   extra_active  gain


###BMR, TDEE, CALORIE GOAL, MAKRO

In [49]:
# KALKULASI BMR, TDEE, CALORIE GOAL, MAKRO
# LABEL / TARGET VARIABLE untuk training NN

synthetic_users['bmr'] = synthetic_users.apply(
    lambda r: calculate_bmr(r['weight_kg'], r['height_cm'], r['age'], r['gender']),
    axis=1
)

synthetic_users['tdee'] = synthetic_users.apply(
    lambda r: calculate_tdee(r['bmr'], r['activity_level']),
    axis=1
)

synthetic_users['calorie_goal'] = synthetic_users.apply(
    lambda r: calculate_calorie_goal(r['tdee'], r['goal']),
    axis=1
)

macro_df = synthetic_users.apply(
    lambda r: pd.Series(calculate_macros(r['calorie_goal'], r['goal'])),
    axis=1
)
synthetic_users = pd.concat([synthetic_users, macro_df], axis=1)

print(synthetic_users[['gender', 'weight_kg', 'height_cm', 'age',
                         'bmr', 'tdee', 'calorie_goal',
                         'protein_target_g', 'fat_target_g',
                         'carbs_target_g']].head(5))

   gender  weight_kg  height_cm  age      bmr     tdee  calorie_goal  \
0    male       86.4      151.9   48  1702.33  2042.80       2342.80   
1  female       79.1      168.2   45  1505.26  1806.31       2106.31   
2    male       68.9      163.4   28  1636.62  3109.58       3409.58   
3    male       74.3      166.6   18  1781.09  2760.69       2760.69   
4    male       85.8      184.4   55  1810.53  2172.64       1672.64   

   protein_target_g  fat_target_g  carbs_target_g  
0             175.7          65.1           263.6  
1             158.0          58.5           237.0  
2             255.7          94.7           383.6  
3             207.1          92.0           276.1  
4             146.4          55.8           146.4  


###ENCODING USER FEATURES

In [50]:
# ENCODING KATEGORIK
# Untuk input layer NN (AI Engineer pakai embedding layer)

le_gender   = LabelEncoder()
le_activity = LabelEncoder()
le_goal     = LabelEncoder()

synthetic_users['gender_encoded']   = le_gender.fit_transform(synthetic_users['gender'])
synthetic_users['activity_encoded'] = le_activity.fit_transform(synthetic_users['activity_level'])
synthetic_users['goal_encoded']     = le_goal.fit_transform(synthetic_users['goal'])

# Simpan mapping untuk dokumentasi
encoding_map = {
    'gender'  : dict(zip(le_gender.classes_,   le_gender.transform(le_gender.classes_))),
    'activity': dict(zip(le_activity.classes_,  le_activity.transform(le_activity.classes_))),
    'goal'    : dict(zip(le_goal.classes_,      le_goal.transform(le_goal.classes_))),
}

print("\nMapping:")
for k, v in encoding_map.items():
    print(f"  {k:10s}: {v}")


Mapping:
  gender    : {'female': np.int64(0), 'male': np.int64(1)}
  activity  : {'extra_active': np.int64(0), 'lightly_active': np.int64(1), 'moderately_active': np.int64(2), 'sedentary': np.int64(3), 'very_active': np.int64(4)}
  goal      : {'gain': np.int64(0), 'lose': np.int64(1), 'maintain': np.int64(2)}


###NORMALISASI FITUR NUMERIK

In [51]:
# NORMALISASI FITUR NUMERIK USER
# MinMaxScaler → range 0-1, cocok untuk NN

user_num_cols = ['weight_kg', 'height_cm', 'age']

user_scaler = MinMaxScaler()
scaled_values = user_scaler.fit_transform(synthetic_users[user_num_cols])

scaled_df = pd.DataFrame(
    scaled_values,
    columns=[f"{c}_scaled" for c in user_num_cols]
)

synthetic_users = pd.concat([synthetic_users, scaled_df], axis=1)

print("✅ Kolom setelah scaling:")
print(synthetic_users.columns.tolist())

✅ Kolom setelah scaling:
['gender', 'weight_kg', 'height_cm', 'age', 'activity_level', 'goal', 'bmr', 'tdee', 'calorie_goal', 'protein_target_g', 'fat_target_g', 'carbs_target_g', 'gender_encoded', 'activity_encoded', 'goal_encoded', 'weight_kg_scaled', 'height_cm_scaled', 'age_scaled']


In [52]:
# --- NORMALIZE COLUMN NAMES ---
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

# --- RENAME KE FORMAT SIMPLE ---
df = df.rename(columns={
    'calories_(kcal_per_100g)': 'calories',
    'protein_(g_per_100g)': 'protein',
    'fat_(g_per_100g)': 'fat',
    'carbohydrates_(g_per_100g)': 'carbohydrates',
    'dietary_fiber_(g_per_100g)': 'fiber',
    'sugars_(g_per_100g)': 'sugar',
    'iron_(mg_per_100g)': 'iron',
    'vitamin_c_(mg_per_100g)': 'vitamin_c',
    'vitamin_b11_(mg_per_100gr)': 'folate',
    'sodium_(mg_per_100g)': 'sodium',
    'calcium_(mg_per_100g)': 'calcium'
})

In [53]:
df = df.rename(columns={'category': 'food_category'})

###FOOD FEATURE ENGINEERING (CORE PART)

In [54]:
# FOOD DATASET — FEATURE ENGINEERING

#Calorie density
df['calorie_density'] = (df['calories'] / 100).round(3)

#Rasio makro dari total kalori
df['calorie_from_protein'] = ((df['protein'] * 4) / df['calories'].replace(0, np.nan)).round(3)
df['calorie_from_fat']     = ((df['fat'] * 9)     / df['calories'].replace(0, np.nan)).round(3)
df['calorie_from_carbs']   = ((df['carbohydrates'] * 4) / df['calories'].replace(0, np.nan)).round(3)

for col in ['calorie_from_protein', 'calorie_from_fat', 'calorie_from_carbs']:
    df[col] = df[col].fillna(0).clip(0, 1)

#Dominant macro profile
def classify_macro_profile(row):
    scores = {
        'high_protein': row['calorie_from_protein'],
        'high_fat'    : row['calorie_from_fat'],
        'high_carb'   : row['calorie_from_carbs']
    }
    return max(scores, key=scores.get)

df['macro_profile'] = df.apply(classify_macro_profile, axis=1)

#Calorie category
def categorize_calories(cal):
    if cal < 100:   return 'very_low'
    elif cal < 250: return 'low'
    elif cal < 400: return 'medium'
    else:           return 'high'

df['calorie_category'] = df['calories'].apply(categorize_calories)

#Protein per 100 kcal
df['protein_per_100kcal'] = (
    (df['protein'] / df['calories'].replace(0, np.nan)) * 100
).round(2).fillna(0)

#High fiber flag
df['is_high_fiber'] = (df['fiber'] >= 5).astype(int)

#Food category encoding
le_food_cat = LabelEncoder()
df['food_category_encoded'] = le_food_cat.fit_transform(
    df['food_category'].fillna('Others')
)

food_cat_map = dict(zip(
    le_food_cat.classes_,
    le_food_cat.transform(le_food_cat.classes_)
))

print(f"   Shape: {df.shape}")

   Shape: (3454, 24)


###VALIDASI HASIL

In [56]:
# VALIDASI HASIL

print("=" * 55)
print("VALIDASI FEATURE ENGINEERING")
print("=" * 55)

# Missing values food dataset
food_fe_cols = [
    'calorie_density', 'calorie_from_protein', 'calorie_from_fat',
    'calorie_from_carbs', 'macro_profile', 'calorie_category',
    'protein_per_100kcal', 'is_high_fiber', 'food_category_encoded'
]
missing = df[food_fe_cols].isnull().sum()
print("\n[1] Missing values food dataset:")
print("  ✅ Tidak ada" if missing.sum() == 0 else missing[missing > 0])

# Range rasio makro
print("\n[2] Range calorie_from_* (harus 0–1):")
for col in ['calorie_from_protein', 'calorie_from_fat', 'calorie_from_carbs']:
    print(f"  {col}: [{df[col].min():.3f}, {df[col].max():.3f}]")

# Hard boundary calorie goal
violations = synthetic_users[synthetic_users['calorie_goal'] < 1200]
print(f"\n[3] Hard boundary calorie_goal >= 1200 kcal:")
print(f"  ✅ Tidak ada pelanggaran" if violations.empty
      else f"  ⚠️  {len(violations)} pelanggaran ditemukan!")

# Distribusi label target
print("\n[4] Distribusi calorie_goal (label utama NN):")
print(f"  Mean : {synthetic_users['calorie_goal'].mean():.1f} kcal")
print(f"  Min  : {synthetic_users['calorie_goal'].min():.1f} kcal")
print(f"  Max  : {synthetic_users['calorie_goal'].max():.1f} kcal")
print(f"  Std  : {synthetic_users['calorie_goal'].std():.1f} kcal")

# Distribusi goal
print("\n[5] Distribusi goal (label kategorik):")
print(synthetic_users['goal'].value_counts().to_string())

VALIDASI FEATURE ENGINEERING

[1] Missing values food dataset:
  ✅ Tidak ada

[2] Range calorie_from_* (harus 0–1):
  calorie_from_protein: [0.000, 1.000]
  calorie_from_fat: [0.000, 1.000]
  calorie_from_carbs: [0.000, 1.000]

[3] Hard boundary calorie_goal >= 1200 kcal:
  ✅ Tidak ada pelanggaran

[4] Distribusi calorie_goal (label utama NN):
  Mean : 2337.8 kcal
  Min  : 1200.0 kcal
  Max  : 3860.3 kcal
  Std  : 567.4 kcal

[5] Distribusi goal (label kategorik):
goal
gain        171
lose        169
maintain    160


In [57]:
# EXPORT FINAL

# --- Output 1: Training data untuk AI Engineer ---
training_cols = [
    # Raw input features (untuk NN input layer)
    'gender', 'weight_kg', 'height_cm', 'age',
    'activity_level', 'goal',
    # Encoded kategorik (untuk embedding layer)
    'gender_encoded', 'activity_encoded', 'goal_encoded',
    # Scaled numerik
    'weight_kg_scaled', 'height_cm_scaled', 'age_scaled',
    # TARGET VARIABLES (yang diprediksi NN)
    'bmr', 'tdee', 'calorie_goal',
    'protein_target_g', 'fat_target_g', 'carbs_target_g'
]

df_training = synthetic_users[training_cols]
df_training.to_csv('training_data_for_ai_engineer.csv', index=False)

# --- Output 2: Food dataset untuk matching layer ---
food_output_cols = [
    'food', 'food_category', 'food_category_encoded',
    'calories', 'protein', 'fat', 'carbohydrates', 'fiber',
    'calorie_density', 'calorie_from_protein', 'calorie_from_fat',
    'calorie_from_carbs', 'macro_profile', 'calorie_category',
    'protein_per_100kcal', 'is_high_fiber'
]
extra_cols  = [c for c in df.columns if c not in food_output_cols]
df_food_final = df[[c for c in food_output_cols + extra_cols if c in df.columns]]
df_food_final.to_csv('featured_nutrition_final.csv', index=False)

In [42]:
from google.colab import files
files.download ("featured_nutrition_final.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [43]:
from google.colab import files
files.download ("training_data_for_ai_engineer.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>